In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix


import warnings
warnings.filterwarnings("ignore")

In [5]:
### Do the one hot encoding for all the categorical data other than city and check for accuracy

In [9]:
### Load Data
df=pd.read_excel("Telco_customer_churn.xlsx")

## Droping Columns
drop_cols = ['CustomerID', 'Count', 'Country', 'State',
       'Lat Long', 'Latitude', 'Longitude',  'Churn Label', 
       'Churn Score', 'CLTV', 'Churn Reason']
df=df.drop(columns=drop_cols)

## Fixing the datatype
df["Total Charges"]=pd.to_numeric(df["Total Charges"],errors="coerce")

## Numerical Columns
num_cols=df.select_dtypes(include=["int","float"]).columns

## Column groups ---
## Generalize for all the categorical values -- nunique greater than 10 do the frequency encoding else do label encoding
def transform(col):
    cat_cols=df.select_dtypes(include=["object","category"]).columns
    for col in cat_cols:
        if df[col].nunique>10:
            freq=df[col].value_counts()
            df[col]=df[col].map(freq)
        else:
            le=LabelEncoder()
            df[col]=le.fit_transform(df[col])
    return df[col]

preprocessor=(transform(col),
              Pipeline([("imputer",KNNImputer(n_neighbors=5))],num_cols)
             )

## Split the data
X=df.drop("Churn Value",axis=1)
y=df["Churn Value"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


xgb_pipeline=Pipeline(steps=[("preprocessing",preprocessor),
                            "model",XGBClassifier(
                                subsample = 0.9, 
                                n_estimators= 300, 
                                min_child_weight=1, 
                                max_depth=7, 
                                learning_rate= 0.05, 
                                gamma=0.1,
                                eval_metric="logloss",
                                scoring="accuracy"         
                            )])
### Train
xgb_pipeline(X_train,y_train)

## prediction
y_pred=xgb_pipeline.predict(X_test)

## Performance
print("Accuracy :",accuracy_score(y_test,y_pred))
print("Confusion Matrix :")
print(confusion_matrix(y_test,y_pred))
print("Classification Report :",classification_report(y_test,y_pred))

NameError: name 'col' is not defined

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

# Load Data
df = pd.read_excel("Telco_customer_churn.xlsx")

# Drop Columns
drop_cols = ['CustomerID', 'Count', 'Country', 'State',
             'Lat Long', 'Latitude', 'Longitude',  
             'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=drop_cols)

# Fix datatype
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

# Split X & y
X = df.drop("Churn Value", axis=1)
y = df["Churn Value"]

# Identify columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns
num_cols = X.select_dtypes(include=["int", "float"]).columns

# Split categorical columns
high_card_cols = [col for col in cat_cols if X[col].nunique() > 10]
low_card_cols = [col for col in cat_cols if X[col].nunique() <= 10]

# Custom Frequency Encoder
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in X.columns:
            self.freq_maps[col] = X[col].value_counts()
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].map(self.freq_maps[col])
        return X

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("freq", FrequencyEncoder(), high_card_cols),
        ("label", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), low_card_cols),
        ("num", KNNImputer(n_neighbors=5), num_cols)
    ]
)

# Pipeline
xgb_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("smote_tomek", SMOTETomek(random_state=42)),
    ("model", XGBClassifier(
        subsample=0.9,
        n_estimators=300,
        min_child_weight=1,
        max_depth=7,
        learning_rate=0.05,
        gamma=0.1,
        eval_metric="logloss",
        random_state=42
    ))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
xgb_pipeline.fit(X_train, y_train)

# Predict
y_pred = xgb_pipeline.predict(X_test)

# Performance
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test, y_pred))

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Accuracy : 0.8055358410220014
Confusion Matrix :
[[893 116]
 [158 242]]
Classification Report :
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1009
           1       0.68      0.60      0.64       400

    accuracy                           0.81      1409
   macro avg       0.76      0.75      0.75      1409
weighted avg       0.80      0.81      0.80      1409

